# Zenith · optional interactive Colab backend
Laptop is primary. Colab is temporary interactive compute, not persistent SaaS hosting. Read `docs/COLAB.md` and the [Colab FAQ](https://research.google.com/colaboratory/faq.html). Free managed runtimes restrict primarily using a separate web UI. Do not enable the optional tunnel unless permitted for your runtime/use. No keepalive tricks. Use consenting demo footage; reset deletes local records. Do not share saved output containing keys.
First generate `artifacts/zenith-colab.zip` locally using `python scripts/package_colab.py`. Upload that code-only package below.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io, os, sys, subprocess, secrets, time
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload only zenith-colab.zip')
root = Path('/content/zenith')
root.mkdir(exist_ok=True)
with ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for member in archive.infolist():
        target = (root / member.filename).resolve()
        if not target.is_relative_to(root) or not member.filename.startswith('backend/'):
            raise ValueError('Unexpected archive path')
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'backend/requirements.txt'), 'pyngrok'], check=True)
os.chdir(root)
print('Backend installed. Restart this setup from a clean runtime if imports conflict.')

In [ ]:
# Set exact browser origin before launching the API. No wildcard CORS.
FRONTEND_ORIGIN = 'http://127.0.0.1:5173'
if 'api_process' in globals() and api_process.poll() is None:
    raise RuntimeError('Backend already running. Run cleanup before restarting.')
operator_key = secrets.token_urlsafe(32)
os.environ['ZENITH_API_KEY'] = operator_key
os.environ['ZENITH_DATA'] = '/content/zenith-data'
os.environ['ZENITH_ORIGINS'] = FRONTEND_ORIGIN
# Preview-only email unless explicitly configured before launch.
api_log = open('/content/zenith-api.log', 'w')
worker_log = open('/content/zenith-worker.log', 'w')
api_process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'backend.main:app', '--host', '127.0.0.1', '--port', '8000'], stdout=api_log, stderr=subprocess.STDOUT)
worker_process = subprocess.Popen([sys.executable, '-m', 'backend.worker'], stdout=worker_log, stderr=subprocess.STDOUT)
print('API PID:', api_process.pid, 'Worker PID:', worker_process.pid)
print('Operator key (private, do not save shared output):', operator_key)
print('If startup fails, inspect /content/zenith-api.log and /content/zenith-worker.log.')

## Notebook-native video analysis
This cell uses the API locally inside the notebook, without a public web UI. Supply a real video and edit normalized polygon coordinates for your scene. Running it creates a real analysis job. Re-run the status cell manually to see progress.

In [ ]:
import requests
headers = {'Authorization': 'Bearer ' + operator_key}
video_upload = files.upload()
filename, content = next(iter(video_upload.items()))
response = requests.post('http://127.0.0.1:8000/api/sources', headers=headers, files={'file': (filename, content)}, timeout=120)
response.raise_for_status()
source = response.json()
from IPython.display import Image, display
preview = requests.get(f"http://127.0.0.1:8000/api/sources/{source['id']}/preview", headers=headers, timeout=30)
preview.raise_for_status()
display(Image(preview.content, width=640))
print('Source:', source['id'], 'Edit the next cell to match this scene.')

In [ ]:
# Edit these for your scene. Never assume a zone from words alone.
POINTS = []  # e.g. [(0.2,0.2), (0.8,0.2), (0.8,0.9), (0.2,0.9)]
CONFIRM_GEOMETRY = False  # Set True only after checking your scene.
if not CONFIRM_GEOMETRY or len(POINTS) < 3:
    raise ValueError('Draw/configure and explicitly confirm your own zone first')
rule = {'capability':'zone','name':'Notebook confirmed zone','points':POINTS,'dwell_seconds':2,'confirmed':True}
response = requests.post('http://127.0.0.1:8000/api/runs', headers=headers, json={'source_id':source['id'],'rule':rule}, timeout=30)
response.raise_for_status()
run_id = response.json()['id']
print('Queued:', run_id)

In [ ]:
response = requests.get(f'http://127.0.0.1:8000/api/runs/{run_id}', headers=headers, timeout=30)
response.raise_for_status()
print(response.json())
events_response = requests.get(f'http://127.0.0.1:8000/api/runs/{run_id}/events', headers=headers, timeout=30)
events_response.raise_for_status()
for event in events_response.json():
    print(event['title'], 'at', event['source_seconds'], 'seconds')
    for frame in event['frames']:
        evidence = requests.get(f"http://127.0.0.1:8000/api/events/{event['id']}/evidence/{frame}", headers=headers, timeout=30)
        evidence.raise_for_status()
        display(Image(evidence.content, width=480))

## Optional authenticated HTTPS tunnel
Only enable where your provider plan and interactive-compute usage permit it. This does not make Colab reliable production hosting. Enter your ngrok account auth token privately. In the local Zenith site, set API origin to the printed HTTPS URL and use the operator key. Keep the exact frontend origin configured above. Never publish the key.

In [ ]:
ENABLE_PUBLIC_TUNNEL = False
if ENABLE_PUBLIC_TUNNEL:
    from getpass import getpass
    from pyngrok import ngrok
    ngrok.set_auth_token(getpass('ngrok auth token: '))
    tunnel = ngrok.connect(addr='127.0.0.1:8000', proto='http', bind_tls=True)
    print('API origin:', tunnel.public_url)
else:
    print('Tunnel disabled. Continue notebook-native interaction or use your laptop backend.')

In [ ]:
# Stop when finished. Download any needed reports before deleting the runtime.
if 'tunnel' in globals():
    ngrok.disconnect(tunnel.public_url)
for process_name in ['api_process', 'worker_process']:
    process = globals().get(process_name)
    if process and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
for stream_name in ['api_log', 'worker_log']:
    stream = globals().get(stream_name)
    if stream:
        stream.close()
print('Stopped.')